# US Foot Traffic Data Download

Downloads Advan/SafeGraph `weekly_patterns` data via Dewey API.

**Key column**: `VISITOR_HOME_CBGS` - JSON dict mapping home census block groups to visitor counts.

**Important**: Download URLs expire quickly. Run `get_file_list()` immediately before `download_files()`.

In [8]:
%cd /workspace

/workspace


In [9]:
import os
from pathlib import Path
from datetime import datetime
import json

import pandas as pd
import deweydatapy as ddp
from dotenv import load_dotenv

# Load .env from current directory
load_dotenv(".env")

print(f"Working directory: {os.getcwd()}")

Working directory: /workspace


In [2]:
# API Key from .env
DEWEY_API_KEY = os.getenv("DEWEY_API_KEY")

if not DEWEY_API_KEY:
    raise ValueError("DEWEY_API_KEY not found. Create .env file with: DEWEY_API_KEY=your_key")

print(f"API Key: {DEWEY_API_KEY[:10]}...{DEWEY_API_KEY[-4:]}")

API Key: akv1_QL_YR...trL-


In [10]:
# Data product URL
WEEKLY_PATTERNS_URL = "https://api.deweydata.io/api/v1/external/data/prj_3a9byed4__fldr_8vyrpa6g3jbudjnsu"

# Output directory (RELATIVE path - ddp requires this)
OUTPUT_DIR = "dbs/us_foot_traffic/weekly_patterns"

# Create directory
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(f"Output: {Path(OUTPUT_DIR).resolve()}")

Output: /workspace/dbs/us_foot_traffic/weekly_patterns


In [11]:
# Date range
START_DATE = "2024-01-01"
END_DATE = "2024-12-31"

print(f"Date range: {START_DATE} to {END_DATE}")

Date range: 2024-01-01 to 2024-12-31


## Check Metadata

In [12]:
meta = ddp.get_meta(DEWEY_API_KEY, WEEKLY_PATTERNS_URL, print_meta=True)

 
Metadata summary ------------------------------------------------
Total number of files: 4,130
Total files size (MB): 1,693,273.63
Date aggregation: DAY
Date partition column: DATE_RANGE_START
Data min available date: 2017-01-02
Data max available date: 2026-02-02
-----------------------------------------------------------------
 


## Download Files

**IMPORTANT**: Run both cells below together - URLs expire quickly!

In [19]:
# Get file list (URLs expire - run immediately before download)
files_df = ddp.get_file_list(
    DEWEY_API_KEY,
    WEEKLY_PATTERNS_URL,
    start_date=START_DATE,
    end_date=END_DATE,
    print_info=True
)

Files information collection completed.
 
Files information summary ---------------------------------------
Total number of pages: 11
Total number of files: 524
Total files size (MB): 218,317.58
Average single file size (MB): 416.1
Date partition column: DATE_RANGE_START
Expires at: 2026-03-11T07:41:41.043004+00:00
-----------------------------------------------------------------


In [20]:
# Download immediately after getting file list
print(f"Downloading {len(files_df)} files to {OUTPUT_DIR}")
print(f"Started: {datetime.now()}")

ddp.download_files(files_df, OUTPUT_DIR, skip_exists=True)

print(f"\nCompleted: {datetime.now()}")

Started: 2026-03-10 07:41:49.649752
File already exists: dbs/us_foot_traffic/weekly_patterns/2024-01-01--data_01c29bf5-0107-dbc6-0042-fa070657a816_004_0_0.snappy.parquet
Skipping...
File already exists: dbs/us_foot_traffic/weekly_patterns/2024-01-01--data_01c29bf5-0107-dbc6-0042-fa070657a816_104_0_0.snappy.parquet
Skipping...
File already exists: dbs/us_foot_traffic/weekly_patterns/2024-01-01--data_01c29bf5-0107-dbc6-0042-fa070657a816_106_0_0.snappy.parquet
Skipping...
File already exists: dbs/us_foot_traffic/weekly_patterns/2024-01-01--data_01c29bf5-0107-dbc6-0042-fa070657a816_106_0_1.snappy.parquet
Skipping...
File already exists: dbs/us_foot_traffic/weekly_patterns/2024-01-01--data_01c29bf5-0107-dbc6-0042-fa070657a816_106_0_2.snappy.parquet
Skipping...
File already exists: dbs/us_foot_traffic/weekly_patterns/2024-01-01--data_01c29bf5-0107-dbc6-0042-fa070657a816_106_0_3.snappy.parquet
Skipping...
File already exists: dbs/us_foot_traffic/weekly_patterns/2024-01-01--data_01c29bf5-0107-

## Verify Download

In [21]:
# Check downloaded files
files = sorted(Path(OUTPUT_DIR).glob("*.parquet"))
print(f"Downloaded files: {len(files)}")

if files:
    # Check file sizes (should be ~400MB each, not 137 bytes)
    for f in files[:5]:
        size_mb = f.stat().st_size / 1024 / 1024
        status = "OK" if size_mb > 1 else "EMPTY/ERROR"
        print(f"  {f.name}: {size_mb:.1f} MB [{status}]")

Downloaded files: 524
  2024-01-01--data_01c29bf5-0107-dbc6-0042-fa070657a816_004_0_0.snappy.parquet: 413.4 MB [OK]
  2024-01-01--data_01c29bf5-0107-dbc6-0042-fa070657a816_104_0_0.snappy.parquet: 415.1 MB [OK]
  2024-01-01--data_01c29bf5-0107-dbc6-0042-fa070657a816_106_0_0.snappy.parquet: 421.9 MB [OK]
  2024-01-01--data_01c29bf5-0107-dbc6-0042-fa070657a816_106_0_1.snappy.parquet: 416.9 MB [OK]
  2024-01-01--data_01c29bf5-0107-dbc6-0042-fa070657a816_106_0_2.snappy.parquet: 423.9 MB [OK]


In [53]:
# Load and check a file
if files:
    # Find a non-empty file
    for f in files:
        if f.stat().st_size > 1000:  # > 1KB
            print(f"Loading: {f.name}")
            df = pd.read_parquet(f)
            print(f"Shape: {df.shape}")
            print(f"\nColumns: {list(df.columns)}")
            break
    else:
        print("All files are empty/corrupted. Check API key and re-run download.")

Loading: 2024-03-04--data_01c29bf5-0107-dbc6-0042-fa070657a816_004_4_0.snappy.parquet
Shape: (527327, 38)

Columns: ['BRAND', 'BUCKETED_DWELL_TIMES', 'CITY', 'CLOSE_DATE', 'DATE_RANGE_END', 'DATE_RANGE_START', 'DEVICE_TYPE', 'DISTANCE_FROM_HOME', 'FOOTPRINT_ID', 'ID_STORE', 'ISO_COUNTRY_CODE', 'IS_DISTRIBUTOR', 'LATITUDE', 'LOCATION_NAME', 'LONGITUDE', 'MEDIAN_DWELL', 'MSA_CODE', 'NAICS_CODE', 'OPEN_DATE', 'PERSISTENT_ID', 'PERSISTENT_ID_STORE', 'POI_CBG', 'POSTAL_CODE', 'REGION', 'RELATED_SAME_DAY_BRAND', 'RELATED_SAME_WEEK_BRAND', 'STREET_ADDRESS', 'SUB_CATEGORY', 'TICKER', 'TOP_CATEGORY', 'VISITOR_COUNTRY_OF_ORIGIN', 'VISITOR_COUNTS', 'VISITOR_DAYTIME_CBGS', 'VISITOR_HOME_AGGREGATION', 'VISITOR_HOME_CBGS', 'VISITS_BY_DAY', 'VISITS_BY_EACH_HOUR', 'VISIT_COUNTS']


## Explore Data

In [54]:
# Sample data
if 'df' in dir() and len(df) > 0:
    key_cols = ['PLACEKEY', 'LOCATION_NAME', 'TOP_CATEGORY', 'SUB_CATEGORY', 
                'LATITUDE', 'LONGITUDE', 'RAW_VISIT_COUNTS']
    key_cols = [c for c in key_cols if c in df.columns]
    display(df[key_cols].head())

,LOCATION_NAME,TOP_CATEGORY,SUB_CATEGORY,LATITUDE,LONGITUDE
0,All Other Specialty Food Stores,Specialty Food Stores,All Other Specialty Food Stores,47.582218,-122.608063
1,Wireless Telecommunications Carriers (except S...,Wired and Wireless Telecommunications Carriers,Wireless Telecommunications Carriers (except S...,43.601289,-89.792853
2,All Other Home Furnishings Stores,Home Furnishings Stores,All Other Home Furnishings Stores,41.221279,-80.636498
3,Electronics Stores,Electronics and Appliance Stores,Electronics Stores,32.901871,-117.207878
4,United States Postal Service,Postal Service,Postal Service,35.465424,-85.661734


In [55]:
# Check VISITOR_HOME_CBGS (key for visitor diversity)
if 'df' in dir() and 'VISITOR_HOME_CBGS' in df.columns:
    print("VISITOR_HOME_CBGS sample (visitor origins):")
    sample = df[df['VISITOR_HOME_CBGS'].notna()].iloc[0]
    cbgs = json.loads(sample['VISITOR_HOME_CBGS']) if isinstance(sample['VISITOR_HOME_CBGS'], str) else sample['VISITOR_HOME_CBGS']
    print(f"POI: {sample.get('LOCATION_NAME', 'N/A')}")
    print(f"# origin CBGs: {len(cbgs)}")
    print(f"Sample: {dict(list(cbgs.items())[:5])}")

VISITOR_HOME_CBGS sample (visitor origins):
POI: All Other Home Furnishings Stores
# origin CBGs: 1
Sample: {'391559312002': 14}


In [56]:
# POI categories
if 'df' in dir() and 'TOP_CATEGORY' in df.columns:
    print("TOP_CATEGORY distribution:")
    for cat, pct in df['TOP_CATEGORY'].value_counts(normalize=True).head(15).items():
        print(f"  {cat}: {pct*100:.1f}%")

TOP_CATEGORY distribution:
  Restaurants and Other Eating Places: 16.4%
  Gasoline Stations : 6.0%
  Other Miscellaneous Store Retailers : 5.2%
  Automotive Repair and Maintenance: 4.8%
  Grocery Stores : 3.9%
  Personal Care Services : 3.4%
  Health and Personal Care Stores : 3.0%
  Activities Related to Credit Intermediation : 2.9%
  Traveler Accommodation: 2.8%
  Clothing Stores : 2.6%
  Automotive Parts, Accessories, and Tire Stores : 2.6%
  Other Amusement and Recreation Industries: 2.5%
  Building Equipment Contractors: 2.4%
  Automobile Dealers : 2.2%
  Building Material and Supplies Dealers : 2.0%


In [57]:
df['TOP_CATEGORY'].nunique()

227